# Small language models for translation
## Pretrained GPT-2 small as a prompting baseline (M0)

**EN.737 · Semester project · M0 and its share of Q1–Q3 (Caio Netto)**

This notebook translates German into English with pretrained GPT-2 small (124M parameters) without updating any weights. GPT-2 reads the team's shared prompt either alone (zero-shot) or after 1, 2, 4, or 8 translated examples from the training split (few-shot). The data selection, prompt, and sacreBLEU settings are those of the M2 notebook, `EN737_Tiny_Transformer_MBR.ipynb`, so M0's numbers can go in the same tables as M1 and M2. The cells compute every reported number.

**First run in Colab.** Choose **Runtime → Change runtime type → GPU** (a T4 is enough) and run the notebook from top to bottom. To check the wiring quickly, set `PROFILE = "smoke"` in the settings cell. The smoke profile also runs on CPU. **Before the project run, edit the prediction cell**, which is saved to disk before anything is scored.

| Section | What it produces |
|---|---|
| Data and model | The M2 splits, GPT-2's parameter count, token counts for German and English, and the KV-cache size per position |
| Choose the baseline | The number of shots for M0, chosen on validation only |
| Score M0 on test | Corpus BLEU and chrF with sacreBLEU signatures, bootstrap intervals, and target cross-entropy for every configuration |
| Q1 baseline | Paired comparisons with zero-shot, and per-sentence outputs for paired tests against M1 |
| Q2 baseline | Scores by source length from a model with no training length cap |
| Q3 baseline | Latency and measured KV-cache memory as demonstrations lengthen the prompt |
| Errors and demo | Error-type rates, a review sheet, and a cell that translates a new sentence |

### Prepare the Colab runtime

Colab already provides PyTorch, `transformers` (which downloads GPT-2 and its tokenizer), `pandas`, and `matplotlib`. This cell installs `sacrebleu` at the M2 notebook's version if it is missing. The settings cell records the versions actually used.

In [ ]:
from importlib import metadata, util
import subprocess
import sys

fallback_versions = {"sacrebleu": "2.5.1", "pyarrow": "20.0.0", "transformers": "4.46.3"}
missing = [f"{name}=={version}" for name, version in fallback_versions.items()
           if util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import csv
import hashlib
import json
import math
import platform
import random
import re
import shutil
import tempfile
import time
import unicodedata
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

### Choose the data, the prompts, and the measurements

**Data.** With `DATA_MODE = "tsv"`, the notebook reads the `shared_splits/` folder that the M2 notebook exports, so all systems use the same rows, IDs, and order. Point `DATA_DIR` at that folder, for example on a mounted Drive, or copy its `train.tsv`, `validation.tsv`, and `test.tsv` into `DATA_DIR` with Colab's **Files** sidebar. With `DATA_MODE = "iwslt"`, it downloads IWSLT 2017 German–English at the M2 notebook's pinned revision and checksums and repeats the M2 selection step. The data section explains when the two modes can differ. The **smoke** profile uses the M2 notebook's synthetic sentences with small settings, and its scores only test the code path.

**Prompts.** `shots` lists the numbers of demonstrations to compare. Each demonstration seed draws one ordered list of training pairs. A $k$-shot prompt uses the first $k$ of them, with the first one closest to the query, so a larger $k$ only adds context to the left of a smaller-$k$ prompt. A demonstration needs at least `demo_min_words` source words and at most `demo_max_tokens` GPT-2 tokens on each side, which keeps 8-shot prompts within GPT-2's 1,024 positions.

**Decoding.** Every configuration uses greedy decoding with at most `max_new_tokens` new tokens and stops at the first newline or end-of-text token.

**Measurements.** A seeded sample of `select_sources` validation sentences chooses the number of shots, and every configuration is then scored on the whole test split. `eval_sources` test sentences, drawn with the M2 notebook's Q3 seed, form the latency and memory benchmark. `bootstrap_samples` resamples of the test sentences give the intervals.

To keep results through a Colab disconnect, set `USE_DRIVE = True` before starting. Finished configurations are then reloaded instead of recomputed.

In [ ]:
PROFILE = "project"                  # "project" or "smoke"
DATA_MODE = "tsv"                    # project mode: "tsv" (M2 shared_splits) or "iwslt"
SOURCE_LANGUAGE = "German"
TARGET_LANGUAGE = "English"
DATA_DIR = "/content/team_data" if Path("/content").is_dir() else "./team_data"
MODEL_ID = "openai-community/gpt2"   # GPT-2 small
MODEL_REVISION = None                # None loads the latest commit; the run records which one
USE_DRIVE = False

if PROFILE not in {"project", "smoke"} or DATA_MODE not in {"iwslt", "tsv"}:
    raise ValueError("Choose one of the profile and data mode options listed above.")

CFG = {
    "data_mode": DATA_MODE, "data_dir": DATA_DIR,
    "source_lang": SOURCE_LANGUAGE, "target_lang": TARGET_LANGUAGE,
    "prompt_template": "{source_lang}: {source}\n{target_lang}:", "demo_separator": "\n\n",
    "split_seed": 737, "max_train": 20_000, "max_validation": 512, "max_test": 512,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "dtype": "float32",
    "shots": [0, 1, 2, 4, 8], "demo_seeds": [101, 202, 303],
    "demo_min_words": 5, "demo_max_tokens": 40,
    "max_new_tokens": 128, "gen_batch_size": 16, "ce_batch_size": 16,
    "select_sources": 128, "select_subset_seed": 4242,
    "eval_sources": 64, "eval_subset_seed": 8021, "timing_repeats": 3,
    "bootstrap_samples": 1000, "bootstrap_seed": 12345,
    "length_edges": [0, 16, 32, 48, 64, None],   # GPT-2 source tokens; None means no upper bound
}
if PROFILE == "smoke":
    CFG.update(data_mode="smoke", source_lang="German", target_lang="English",
               max_train=None, max_validation=None, max_test=None,
               shots=[0, 2], demo_seeds=[101, 202], demo_min_words=1, max_new_tokens=24,
               gen_batch_size=4, ce_batch_size=4, select_sources=6, eval_sources=4,
               timing_repeats=1, bootstrap_samples=200, length_edges=[0, 8, None])

shots = CFG["shots"]
if 0 not in shots or not any(k > 0 for k in shots) or len(set(shots)) != len(shots) or min(shots) < 0:
    raise ValueError("List distinct shot counts that include 0 and at least one positive count.")
CFG["shots"] = sorted(shots)
if not CFG["demo_seeds"] or len(set(CFG["demo_seeds"])) != len(CFG["demo_seeds"]):
    raise ValueError("Supply at least one demonstration seed, without repeats.")
positive = ("max_new_tokens", "gen_batch_size", "ce_batch_size", "select_sources", "eval_sources",
            "timing_repeats", "bootstrap_samples", "demo_max_tokens")
if min(CFG[key] for key in positive) < 1:
    raise ValueError("Sizes, limits, and repeat counts must be positive.")
finite_edges = CFG["length_edges"][:-1]
if CFG["length_edges"][-1] is not None or any(b <= a for a, b in zip(finite_edges, finite_edges[1:])):
    raise ValueError("length_edges must increase and end with None.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))
MODEL_DTYPE = {"float32": torch.float32, "float16": torch.float16,
               "bfloat16": torch.bfloat16}[CFG["dtype"]]


def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


versions = {name: package_version(name) for name in
            ["torch", "transformers", "tokenizers", "safetensors", "numpy", "pandas",
             "matplotlib", "pyarrow", "sacrebleu"]}
hardware = {"device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
            "cuda": torch.version.cuda, "python": platform.python_version(),
            "platform": platform.platform()}
print(f"Profile: {PROFILE} | Device: {hardware['gpu'] or DEVICE} | dtype: {CFG['dtype']}")
if PROFILE == "project" and DEVICE.type == "cpu":
    print("A GPU is recommended for the project profile. The smoke profile also works on CPU.")

### Record the predictions before scoring

The project asks for a hypothesis and a quantitative prediction before the main experiments. M0 supplies the baseline side of Q1–Q3, so its predictions are about the baseline. **The values below are a starting point. Replace them with the numbers the team expects before the first project run.** The next cell saves them to `prediction_before_scoring.json` with a UTC timestamp before any translation is scored. They also enter the run's identity, so editing them later starts a new run folder instead of relabeling old results. A later cell checks each prediction against what was observed, whichever way the result goes.

| | Hypothesis | Prediction |
|---|---|---|
| H1 | GPT-2 saw little German in pretraining, so it translates poorly without examples. | Zero-shot test chrF ≤ 25 and BLEU ≤ 3. |
| H2 | Demonstrations show the model the task (Q1 baseline). | The few-shot configuration chosen on validation beats zero-shot by ≥ 3 chrF points on test, averaged over demonstration seeds. |
| H3 | Demonstrations make the human translation more probable. | Test target cross-entropy falls by ≥ 0.3 nats per token from 0 shots to the largest shot count. |
| H4 | Cross-entropy tracks translation quality. | In the chosen configuration, sentence-level cross-entropy and sentence chrF have a Spearman correlation ≤ −0.2. |
| H5 | Demonstrations teach the model to stop after one line, and the cache follows its formula (Q3 baseline). | Mean generated tokens fall by ≥ 30% from 0 shots to the smallest positive shot count. On a GPU, the memory held after generation is within 10% of the KV-cache prediction. |

In [ ]:
PREDICTIONS = {
    "H1_zero_shot_chrf_max": 25.0,
    "H1_zero_shot_bleu_max": 3.0,
    "H2_min_chrf_gain_over_zero_shot": 3.0,
    "H3_min_ce_drop_nats_per_token": 0.3,
    "H4_max_spearman_ce_vs_chrf": -0.2,
    "H5_min_generated_token_reduction": 0.30,
    "H5_held_memory_tolerance": 0.10,
}
PLAN = {
    "system": "M0: pretrained GPT-2 small, frozen weights, greedy decoding, newline stop",
    "configurations": "0 shots once; every positive shot count once per demonstration seed",
    "baseline_choice": "highest mean validation chrF over demonstration seeds; ties go to fewer shots",
    "controls": ["same test IDs and references for every configuration",
                 "demonstrations drawn from the training split only",
                 "nested demonstration lists within a seed",
                 "same template, decoder, generation cap and stop rule",
                 "same weights, dtype, batch size and padding side"],
    "repeats": {"demonstration_seeds": CFG["demo_seeds"], "timing_repeats": CFG["timing_repeats"],
                "bootstrap_samples": CFG["bootstrap_samples"]},
}


def write_json(path, obj):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


identity_data = {"profile": PROFILE, "config": CFG, "predictions": PREDICTIONS}
if CFG["data_mode"] == "tsv":
    file_hashes = {}
    for split in ("train", "validation", "test"):
        path = Path(CFG["data_dir"]) / f"{split}.tsv"
        if not path.is_file():
            raise FileNotFoundError(
                f"Place {split}.tsv from the M2 run's shared_splits folder in {CFG['data_dir']}, "
                "or set DATA_MODE = 'iwslt'.")
        file_hashes[split] = file_sha256(path)
    identity_data["tsv_sha256"] = file_hashes

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = Path("/content/drive/MyDrive/en737_m0")
else:
    RUNS = (Path("/content") if Path("/content").is_dir() else Path.cwd()) / "en737_m0_runs"
run_key = hashlib.sha256(json.dumps(identity_data, sort_keys=True).encode()).hexdigest()[:12]
OUT = RUNS / f"{PROFILE}-{run_key}"
OUT.mkdir(parents=True, exist_ok=True)

prediction_path = OUT / "prediction_before_scoring.json"
if not prediction_path.exists():
    write_json(prediction_path, {
        "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "predictions": PREDICTIONS, "plan": PLAN})
RECORDED = json.loads(prediction_path.read_text(encoding="utf-8"))
write_json(OUT / "settings.json", {**identity_data, "versions": versions, "hardware": hardware})
print(f"Outputs: {OUT}")
print(f"Predictions recorded at {RECORDED['recorded_utc']} (UTC)")

## Data

### Load the parallel sentences

These cells repeat the M2 notebook's data code so that both notebooks select the same sentences. `tsv` reads the three files that M2 wrote. `iwslt` downloads the **German → English** IWSLT 2017 release at the pinned revision, checks each file's SHA-256, and keeps the published train, validation, and test boundaries ([IWSLT 2017 files](https://huggingface.co/datasets/IWSLT/iwslt2017/tree/b12a1d9c35c86684cfb15079c9a2fbb467a23aa1/iwslt2017-de-en)). `smoke` builds the M2 notebook's short synthetic sentences.

The M2 notebook writes `shared_splits/` inside its run folder, for example `/content/en737_m2_runs/project-<key>/shared_splits/`, in the code cell under "Define the training cohort and preserve complete references". That cell runs before any training, so the M2 notebook only has to run down to it, which takes a few minutes. If M2's `q3_test_subset.tsv`, which is in the same run folder, is also copied into `DATA_DIR`, the selection cell below confirms that the latency benchmark uses the same sentences as M2's Q3(a) study.

In [ ]:
SPLIT_NAMES = ("train", "validation", "test")
IWSLT_REVISION = "b12a1d9c35c86684cfb15079c9a2fbb467a23aa1"
IWSLT_SHA256 = {
    "train": "775169702cc226a66b768ab7f3fb22b744ce5a6e037b677e068a9d51c4af9fce",
    "validation": "6c4385add06f6d19de34978cc481e59a1f1b354ceaa2247101629f15c351b6f3",
    "test": "2ed8e9880200bd7adddd9e56664f8607fb1dbbaff18f00bd811ac8378c4e4b94",
}


def read_iwslt(split):
    import pyarrow.parquet as pq

    folder = Path(CFG["data_dir"]) / "iwslt2017" / IWSLT_REVISION
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"iwslt2017-{split}.parquet"
    if not path.exists() or file_sha256(path) != IWSLT_SHA256[split]:
        url = (f"https://huggingface.co/datasets/IWSLT/iwslt2017/resolve/"
               f"{IWSLT_REVISION}/iwslt2017-de-en/{path.name}")
        print(f"Fetching IWSLT {split} ...")
        with tempfile.NamedTemporaryFile(dir=folder, suffix=".part", delete=False) as tmp:
            tmp_path = Path(tmp.name)
        try:
            with urllib.request.urlopen(url, timeout=120) as response, tmp_path.open("wb") as dest:
                shutil.copyfileobj(response, dest)
            if file_sha256(tmp_path) != IWSLT_SHA256[split]:
                raise ValueError(f"Checksum mismatch for {split}. Rerun the download cell.")
            tmp_path.replace(path)
        finally:
            tmp_path.unlink(missing_ok=True)
    translations = pq.read_table(path, columns=["translation"]).column("translation").to_pylist()
    return [(str(i), pair["de"], pair["en"]) for i, pair in enumerate(translations)]


def read_tsv(split):
    path = Path(CFG["data_dir"]) / f"{split}.tsv"
    with path.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        if not {"source", "target"}.issubset(reader.fieldnames or []):
            raise ValueError(f"{path} needs tab-separated source and target column headers.")
        rows = []
        for i, row in enumerate(reader):
            if None in row or row["source"] is None or row["target"] is None:
                raise ValueError(f"Malformed TSV row {i + 2} in {path}. Quote embedded tabs.")
            row_id = row.get("id")
            rows.append((str(i) if not row_id else row_id, row["source"], row["target"]))
        return rows


def smoke_splits():
    names = ["Anna", "Ben", "Clara", "David", "Emma", "Felix"]
    verbs = [("kauft", "buys"), ("sieht", "sees"), ("braucht", "needs"), ("mag", "likes")]
    foods = [("Brot", "bread"), ("Reis", "rice"), ("Milch", "milk"), ("Tee", "tea")]
    pairs = [(f"{name} {de_v} {de_n}.", f"{name} {en_v} {en_n}.")
             for name in names for de_v, en_v in verbs for de_n, en_n in foods]
    random.Random(4309).shuffle(pairs)
    parts = (pairs[:72], pairs[72:84], pairs[84:])
    return {split: [(str(i), s, t) for i, (s, t) in enumerate(part)]
            for split, part in zip(SPLIT_NAMES, parts)}


mode = CFG["data_mode"]
if mode in ("iwslt", "smoke") and (CFG["source_lang"], CFG["target_lang"]) != ("German", "English"):
    raise ValueError("The built-in datasets use German → English. Choose tsv for another pair.")
if mode == "smoke":
    raw_splits = smoke_splits()
elif mode == "iwslt":
    raw_splits = {split: read_iwslt(split) for split in SPLIT_NAMES}
elif mode == "tsv":
    raw_splits = {split: read_tsv(split) for split in SPLIT_NAMES}
else:
    raise ValueError("Choose data_mode = 'iwslt', 'tsv', or 'smoke'.")
print("Loaded rows:", {name: len(rows) for name, rows in raw_splits.items()})

### Apply the M2 selection rules

For `iwslt` and `smoke`, this cell runs the M2 selection step unchanged. Unicode is normalized to NFC and whitespace is collapsed. Only the first translation of a repeated source is kept. Validation drops sources already selected for training, and test drops sources selected for either earlier split. Each split is shuffled with seed `split_seed + offset`, and the first 20,000, 512, and 512 rows are kept. M2 then removes rows whose prompt exceeds 128 of its own tokens, and its `data_report.json` counts them as `prompt_too_long`. This notebook cannot repeat that step without M2's tokenizer. `tsv` mode reads the rows that remain after it, so `tsv` matches M2 exactly and `iwslt` differs only by those rows. In `tsv` mode the cell keeps the file order, normalizes the text, and stops if it finds empty, repeated, or overlapping sources.

The test split is the evaluation set for every configuration. A seeded validation sample chooses the number of shots, and demonstrations come only from training.

In [ ]:
def clean_sentence(text):
    if not isinstance(text, str):
        raise ValueError(f"Expected a text sentence, received {type(text).__name__}.")
    return " ".join(unicodedata.normalize("NFC", text).split())


def prompt_text(source):
    return CFG["prompt_template"].format(
        source_lang=CFG["source_lang"], source=source, target_lang=CFG["target_lang"])


splits = {}
data_report = {"data_mode": mode, "splits": {}}
if mode == "iwslt":
    data_report.update(iwslt_revision=IWSLT_REVISION, parquet_sha256=IWSLT_SHA256)
if mode == "tsv":
    data_report["tsv_sha256"] = identity_data["tsv_sha256"]
seen_earlier = set()
for offset, split in enumerate(SPLIT_NAMES):
    candidates, seen_here, seen_ids = [], set(), set()
    stats = {"raw_rows": len(raw_splits[split]), "empty": 0,
             "repeated_source": 0, "source_in_earlier_split": 0}
    for row_id, source, target in raw_splits[split]:
        source, target = clean_sentence(source), clean_sentence(target)
        if not source or not target:
            stats["empty"] += 1
            continue
        if source in seen_earlier:
            stats["source_in_earlier_split"] += 1
            continue
        if source in seen_here:
            stats["repeated_source"] += 1
            continue
        if row_id in seen_ids:
            raise ValueError(f"Duplicate id {row_id!r} in {split}. Give each source a unique id.")
        seen_here.add(source)
        seen_ids.add(row_id)
        candidates.append({"id": row_id, "source": source, "target": target})
    if mode == "tsv":
        dropped = stats["empty"] + stats["repeated_source"] + stats["source_in_earlier_split"]
        if dropped:
            raise ValueError(f"{split}.tsv has {dropped} empty, repeated, or overlapping rows. "
                             "tsv mode expects already selected splits, such as M2's shared_splits.")
        rows = candidates
    else:
        random.Random(CFG["split_seed"] + offset).shuffle(candidates)
        limit = CFG[f"max_{split}"]
        if limit is not None and limit < 1:
            raise ValueError(f"max_{split} must be positive or None.")
        rows = candidates[:limit]
    if not rows:
        raise ValueError(f"No examples available in {split} after cleaning and source checks.")
    splits[split] = rows
    stats.update(eligible_before_cap=len(candidates), selected=len(rows))
    data_report["splits"][split] = stats
    seen_earlier.update(row["source"] for row in rows)
del raw_splits, candidates, seen_earlier, seen_here, seen_ids

digest = hashlib.sha256()
for split in SPLIT_NAMES:
    for row in splits[split]:
        record = [split, row["id"], row["source"], row["target"]]
        digest.update((json.dumps(record, ensure_ascii=False) + "\n").encode())
DATA_SIGNATURE = digest.hexdigest()
data_report["data_signature"] = DATA_SIGNATURE

m2_subset_path = Path(CFG["data_dir"]) / "q3_test_subset.tsv"
if mode == "tsv" and m2_subset_path.is_file():
    m2_ids = pd.read_csv(m2_subset_path, sep="\t", dtype=str, keep_default_na=False)["id"].tolist()
    ours = random.Random(CFG["eval_subset_seed"]).sample(
        splits["test"], min(CFG["eval_sources"], len(splits["test"])))
    data_report["q3_subset_matches_m2"] = m2_ids == [row["id"] for row in ours]
    print("Benchmark sentences match M2's q3_test_subset.tsv:", data_report["q3_subset_matches_m2"])
write_json(OUT / "data_report.json", data_report)
for split in SPLIT_NAMES:
    print(f"{split:10} rows={len(splits[split]):6}")
print("Data SHA-256:", DATA_SIGNATURE)
print("Prompt for the first test source:", repr(prompt_text(splits["test"][0]["source"])))

## Model

### Load GPT-2 small and its tokenizer

GPT-2 small is a 12-layer decoder-only Transformer of width 768 with 12 attention heads, a 50,257-token byte-level BPE vocabulary, and 1,024 learned positions. Its output projection reuses the input embedding matrix, and the parameter count counts that matrix once. The weights stay frozen in evaluation mode throughout. `model_info.json` records the Hugging Face commit that was loaded, and `MODEL_REVISION` can pin it later.

**Stopping.** Nothing in pretraining teaches GPT-2 to emit end-of-text after a translation. The shared prompt puts the source and the target on separate lines, so decoding stops at the first token containing a newline, or at end-of-text. The stop list holds every vocabulary entry whose text contains a newline. Apart from trimming spaces, the output is not edited.

**KV cache.** During generation, every layer stores one key vector and one value vector of width $d_\text{model}$ for each cached position. A sequence of $L$ positions therefore holds $2\,n_\text{layer}\,d_\text{model}\,L$ numbers. For GPT-2 small that is 18,432 numbers per position: 72 KiB in float32, the dtype used here, and 36 KiB in float16.

**Token counts.** GPT-2's vocabulary was learned on English web text, so German words split into more pieces. The table compares tokens per word for the German sources and the English references of the test split. Longer German prompts cost context positions and cache memory.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_id"], revision=CFG["model_revision"])
model = AutoModelForCausalLM.from_pretrained(CFG["model_id"], revision=CFG["model_revision"])
model = model.to(device=DEVICE, dtype=MODEL_DTYPE).eval()
model.requires_grad_(False)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
EOS_ID, PAD_ID = tokenizer.eos_token_id, tokenizer.pad_token_id
N_POSITIONS = getattr(model.config, "n_positions", None) or model.config.max_position_embeddings
N_LAYER, D_MODEL = model.config.num_hidden_layers, model.config.hidden_size
BYTES_PER_VALUE = torch.finfo(MODEL_DTYPE).bits // 8
KV_BYTES_PER_POSITION = 2 * N_LAYER * D_MODEL * BYTES_PER_VALUE
PARAMS = sum(p.numel() for p in model.parameters())
COMMIT = getattr(model.config, "_commit_hash", None)
if CFG["model_id"] in {"openai-community/gpt2", "gpt2"} and PARAMS != 124_439_808:
    raise ValueError(f"Expected 124,439,808 parameters for GPT-2 small, found {PARAMS:,}.")

if COMMIT is None:
    print("Could not read the model commit. Pin MODEL_REVISION to a commit hash for exact reproducibility.")

# Every vocabulary entry whose text contains a newline ends a translation, as does end-of-text.
vocab_text = [tokenizer.decode([i], clean_up_tokenization_spaces=False) for i in range(len(tokenizer))]
NEWLINE_IDS = sorted(i for i, text in enumerate(vocab_text) if "\n" in text)
STOP_IDS = sorted(set(NEWLINE_IDS) | {EOS_ID})
NEWLINE_SET, STOP_SET = set(NEWLINE_IDS), set(STOP_IDS)
del vocab_text

for split, rows in splits.items():
    source_ids = tokenizer([row["source"] for row in rows], add_special_tokens=False)["input_ids"]
    target_ids = tokenizer([" " + row["target"] for row in rows], add_special_tokens=False)["input_ids"]
    for row, s_ids, t_ids in zip(rows, source_ids, target_ids):
        row["source_tokens"] = len(s_ids)
        row["target_ids"] = t_ids

lengths = pd.DataFrame({
    "source_words": [len(r["source"].split()) for r in splits["test"]],
    "source_tokens": [r["source_tokens"] for r in splits["test"]],
    "reference_words": [len(r["target"].split()) for r in splits["test"]],
    "reference_tokens": [len(r["target_ids"]) for r in splits["test"]],
})
token_table = pd.DataFrame({
    "German sources": [lengths.source_words.mean(), lengths.source_tokens.mean(),
                       lengths.source_tokens.sum() / lengths.source_words.sum()],
    "English references": [lengths.reference_words.mean(), lengths.reference_tokens.mean(),
                           lengths.reference_tokens.sum() / lengths.reference_words.sum()],
}, index=["mean words", "mean GPT-2 tokens", "GPT-2 tokens per word"])
over_cap = float((lengths.reference_tokens > CFG["max_new_tokens"]).mean())

MODEL_INFO = {
    "model_id": CFG["model_id"], "requested_revision": CFG["model_revision"], "loaded_commit": COMMIT,
    "parameters": PARAMS, "n_layer": N_LAYER, "d_model": D_MODEL, "n_positions": N_POSITIONS,
    "vocab_size": len(tokenizer), "dtype": CFG["dtype"],
    "attention": getattr(model.config, "_attn_implementation", None),
    "kv_bytes_per_position": KV_BYTES_PER_POSITION, "stop_token_ids": len(STOP_IDS),
}
write_json(OUT / "model_info.json", MODEL_INFO)
data_report["gpt2_token_counts_test"] = {column: {index: float(value) for index, value in values.items()}
                                         for column, values in token_table.items()}
data_report["test_references_over_cap"] = over_cap
write_json(OUT / "data_report.json", data_report)

print(f"{CFG['model_id']} at commit {COMMIT} | {PARAMS:,} parameters | "
      f"{N_LAYER} layers, width {D_MODEL}, {N_POSITIONS} positions")
print(f"KV cache: {KV_BYTES_PER_POSITION:,} bytes ({KV_BYTES_PER_POSITION / 1024:.0f} KiB) "
      f"per cached position in {CFG['dtype']}")
print(f"Stop list: {len(STOP_IDS)} token IDs (end-of-text and every entry containing a newline)")
display(token_table.round(2))
print(f"Test references longer than the {CFG['max_new_tokens']}-token cap: {100 * over_cap:.2f}%")

### Build the prompts

The query prompt is the shared template, `German: {source}` with `English:` on the next line, and no beginning-of-text token is added. A demonstration is the same template followed by a space and the reference, and a blank line separates demonstrations. For each seed, the cell draws `max(shots)` training pairs once, in order, from the pairs that satisfy the length rules. A $k$-shot prompt lists the first $k$ of them, with the first nearest the query. For example, the 8-shot prompt is the 4-shot prompt with four more demonstrations in front.

Every prompt must leave room for the generation cap and for its reference, which teacher forcing appends. If a long source does not fit, the cell drops demonstrations starting with the farthest from the query and records how many remain. The run tables count the affected sentences, which should be rare with the default settings.

In [ ]:
def demonstration_text(row):
    return prompt_text(row["source"]) + " " + row["target"]


def build_prompt(source, demos, reserve):
    """Token IDs with demos[0] nearest the query, dropping the farthest demonstrations if needed."""
    used = list(demos)
    while True:
        context = "".join(demonstration_text(d) + CFG["demo_separator"] for d in reversed(used))
        ids = tokenizer(context + prompt_text(source), add_special_tokens=False)["input_ids"]
        if len(ids) + reserve <= N_POSITIONS or not used:
            break
        used = used[:-1]
    if len(ids) + reserve > N_POSITIONS:
        raise ValueError(f"The prompt has {len(ids)} tokens and leaves no room for {reserve} more "
                         f"within {N_POSITIONS} positions.")
    return ids, len(used)


demo_pool = [row for row in splits["train"]
             if len(row["source"].split()) >= CFG["demo_min_words"]
             and row["source_tokens"] <= CFG["demo_max_tokens"]
             and len(row["target_ids"]) <= CFG["demo_max_tokens"]]
MAX_SHOTS = max(CFG["shots"])
if len(demo_pool) < MAX_SHOTS:
    raise ValueError(f"Only {len(demo_pool)} training pairs satisfy the demonstration rules.")
DEMOS = {seed: random.Random(seed).sample(demo_pool, MAX_SHOTS) for seed in CFG["demo_seeds"]}
write_json(OUT / "demonstrations.json",
           {str(seed): [{key: d[key] for key in ("id", "source", "target")} for d in demos]
            for seed, demos in DEMOS.items()})

example_shots = min(k for k in CFG["shots"] if k > 0)
example_ids, _ = build_prompt(splits["validation"][0]["source"],
                              DEMOS[CFG["demo_seeds"][0]][:example_shots], CFG["max_new_tokens"])
demo_lengths = [len(tokenizer(demonstration_text(d) + CFG["demo_separator"],
                              add_special_tokens=False)["input_ids"])
                for demos in DEMOS.values() for d in demos]
print(f"Demonstration pool: {len(demo_pool):,} of {len(splits['train']):,} training pairs | "
      f"mean demonstration length {np.mean(demo_lengths):.1f} GPT-2 tokens")
print(f"Example {example_shots}-shot prompt ({len(example_ids)} tokens):\n")
print(tokenizer.decode(example_ids))

### Generate and score references

`generate_batch` left-pads a batch of prompts and calls Hugging Face greedy generation with the stop list. Hugging Face derives position IDs from the attention mask, so the padding does not shift positions. The translation is the decoded text before the first stop token, with outer spaces removed, and `stop_reason` records whether a newline, end-of-text, or the cap ended it. `measured_generate` decodes one prompt at a time for the benchmark section and also returns its latency and GPU memory.

`target_nll` returns the teacher-forced negative log-likelihood of each reference, $-\sum_j \log p_\theta(y_j \mid \text{prompt}, y_{<j})$, in nats. The reference tokens are those of a space followed by the reference text. No end-of-text token is added, since pretrained GPT-2 is not asked to produce one. When M1 is compared with M0 in this unit, M1 should be scored on the same tokens. As in the M2 notebook, logits are computed only at the positions that predict reference tokens. Cross-entropy per token is total NLL divided by total target tokens. The tables also give bits per UTF-8 byte of the target text, a unit that does not depend on the tokenizer.

In [ ]:
def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


def finish(new_ids):
    """Cut a continuation at its first stop token and decode the translation."""
    cut = next((j for j, token in enumerate(new_ids) if token in STOP_SET), None)
    body = new_ids if cut is None else new_ids[:cut]
    if cut is None:
        reason = "cap"
    elif new_ids[cut] in NEWLINE_SET:
        reason = "newline"
    else:
        reason = "eos"
    text = tokenizer.decode(body, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return {"prediction": text.split("\n", 1)[0].strip(), "stop_reason": reason,
            "generated_tokens": len(body) + (cut is not None)}


GENERATION_ARGS = dict(do_sample=False, num_beams=1, eos_token_id=STOP_IDS, pad_token_id=PAD_ID)


def left_pad(id_lists):
    width = max(map(len, id_lists))
    ids = torch.full((len(id_lists), width), PAD_ID, dtype=torch.long)
    mask = torch.zeros_like(ids)
    for i, row in enumerate(id_lists):
        ids[i, width - len(row):] = torch.tensor(row, dtype=torch.long)
        mask[i, width - len(row):] = 1
    return ids.to(DEVICE), mask.to(DEVICE)


@torch.inference_mode()
def generate_batch(id_lists, max_new_tokens):
    ids, mask = left_pad(id_lists)
    out = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=max_new_tokens,
                         **GENERATION_ARGS)
    return [finish(row) for row in out[:, ids.shape[1]:].tolist()]


def gpu_bytes(kind):
    """Bytes of live tensors ("current") or their peak since the last reset ("peak").

    PyTorch's requested_bytes counters count tensor sizes without the caching allocator's
    block rounding, which can add up to 1 MiB per large tensor to memory_allocated().
    """
    stats = torch.cuda.memory_stats(DEVICE)
    return stats.get(f"requested_bytes.all.{kind}", stats[f"allocated_bytes.all.{kind}"])


@torch.inference_mode()
def measured_generate(prompt_ids):
    """Decode one prompt; return the result, seconds, and held and peak GPU bytes (NaN on CPU)."""
    ids = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    mask = torch.ones_like(ids)
    on_gpu = DEVICE.type == "cuda"
    synchronize()
    if on_gpu:
        torch.cuda.reset_peak_memory_stats(DEVICE)
        before = gpu_bytes("current")
    start = time.perf_counter()
    out = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=CFG["max_new_tokens"],
                         return_dict_in_generate=True, **GENERATION_ARGS)
    synchronize()
    held = peak = float("nan")
    if on_gpu:
        # The returned output still holds the KV cache at this point.
        held = gpu_bytes("current") - before
        peak = gpu_bytes("peak") - before
    result = finish(out.sequences[0, ids.shape[1]:].tolist())
    seconds = time.perf_counter() - start
    del out
    return result, seconds, held, peak


def ce_batch(prompt_lists, target_lists):
    width = max(len(p) + len(t) for p, t in zip(prompt_lists, target_lists))
    ids = torch.full((len(prompt_lists), width), PAD_ID, dtype=torch.long)
    mask = torch.zeros_like(ids)
    labels = torch.full_like(ids, -100)
    for i, (prompt, target) in enumerate(zip(prompt_lists, target_lists)):
        stop = len(prompt) + len(target)
        ids[i, :stop] = torch.tensor(prompt + target, dtype=torch.long)
        mask[i, :stop] = 1
        # The hidden state at position j predicts the token at position j + 1.
        labels[i, len(prompt) - 1:stop - 1] = torch.tensor(target, dtype=torch.long)
    return ids, mask, labels


@torch.inference_mode()
def target_nll(prompt_lists, target_lists):
    """Teacher-forced NLL of each target sequence given its prompt, in nats."""
    head = model.get_output_embeddings()
    order = sorted(range(len(prompt_lists)), key=lambda i: len(prompt_lists[i]) + len(target_lists[i]))
    nll = [0.0] * len(order)
    for start in range(0, len(order), CFG["ce_batch_size"]):
        batch = order[start:start + CFG["ce_batch_size"]]
        ids, mask, labels = (x.to(DEVICE) for x in ce_batch([prompt_lists[i] for i in batch],
                                                             [target_lists[i] for i in batch]))
        hidden = model.base_model(input_ids=ids, attention_mask=mask, use_cache=False).last_hidden_state
        keep = labels.ne(-100)
        token_nll = F.cross_entropy(head(hidden[keep]).float(), labels[keep], reduction="none")
        sums = torch.zeros(len(batch), dtype=torch.float64, device=DEVICE)
        sums.index_add_(0, keep.nonzero()[:, 0], token_nll.double())
        for i, value in zip(batch, sums.tolist()):
            nll[i] = value
    return nll

### Score with sacreBLEU and resample sentences

BLEU and chrF use the M2 notebook's sacreBLEU settings, and every corpus score comes with its sacreBLEU signature. The notebook also keeps sacreBLEU's per-sentence sufficient statistics. Summing them gives the corpus score of any subset or bootstrap resample without re-tokenizing, and each corpus score is checked against sacreBLEU's own `corpus_score`. All configurations on a split share one matrix of resampling counts, so differences between configurations are paired bootstrap comparisons. Intervals are 95% percentile intervals.

In [ ]:
BLEU_METRIC = BLEU(tokenize="13a", lowercase=False, effective_order=False)
CHRF_METRIC = CHRF(char_order=6, word_order=0, beta=2)
METRICS = {"bleu": BLEU_METRIC, "chrf": CHRF_METRIC}


def score_from_stats(metric, summed):
    return float(metric._compute_score_from_stats(summed).score)


def corpus_scores(hypotheses, references):
    """Corpus scores, per-sentence sufficient statistics, and signatures for BLEU and chrF."""
    result = {}
    for name, metric in METRICS.items():
        stats = np.asarray(metric._extract_corpus_statistics(hypotheses, [references]), dtype=np.float64)
        official = float(metric.corpus_score(hypotheses, [references]).score)
        summed = score_from_stats(metric, stats.sum(axis=0))
        if not math.isclose(summed, official, rel_tol=1e-6, abs_tol=1e-6):
            raise RuntimeError(f"{name}: summed statistics give {summed}, sacreBLEU gives {official}.")
        result[name] = {"score": official, "stats": stats, "signature": str(metric.get_signature())}
    return result


def bootstrap_counts(n, samples, seed):
    """How many times each of n sentences appears in each bootstrap resample."""
    rng = np.random.default_rng(seed)
    counts = np.zeros((samples, n))
    np.add.at(counts, (np.arange(samples)[:, None], rng.integers(0, n, size=(samples, n))), 1.0)
    return counts


def bootstrap_scores(metric, stats, counts):
    return np.array([score_from_stats(metric, row) for row in counts @ stats])


def interval(values):
    low, high = np.percentile(values, [2.5, 97.5])
    return float(low), float(high)

### Check the pipeline

These checks take seconds and stop the notebook if an assumption fails.

1. Tokenizing a prompt and its reference separately gives the same tokens as tokenizing them together, so the teacher-forced sequence is the one GPT-2 would read.
2. Cross-entropy labels sit one position before their tokens, and padding never enters the loss.
3. A batch of references gets the same NLL as one reference at a time, and the NLL matches the loss that Hugging Face computes for the same sequence.
4. The stop rule and the post-processing behave as described on hand-made token lists.
5. Left-padded batched generation gives the same translations as one prompt at a time. A single difference can come from a near-tied token, so the check stops the notebook only when two or more prompts differ.

In [ ]:
for row in splits["train"][:20]:
    joint = tokenizer(prompt_text(row["source"]) + " " + row["target"],
                      add_special_tokens=False)["input_ids"]
    separate = (tokenizer(prompt_text(row["source"]), add_special_tokens=False)["input_ids"]
                + tokenizer(" " + row["target"], add_special_tokens=False)["input_ids"])
    assert joint == separate, f"Tokens merge across the prompt boundary for training id {row['id']}."

_, toy_mask, toy_labels = ce_batch([[11, 12, 13], [21]], [[14, 15], [22, 23, 24]])
assert toy_labels.tolist() == [[-100, -100, 14, 15, -100], [22, 23, 24, -100, -100]]
assert toy_mask.tolist() == [[1, 1, 1, 1, 1], [1, 1, 1, 1, 0]]

probe_rows = splits["validation"][:min(4, len(splits["validation"]))]
probe_prompts = [build_prompt(r["source"], [], max(CFG["max_new_tokens"], len(r["target_ids"])))[0]
                 for r in probe_rows]
probe_targets = [r["target_ids"] for r in probe_rows]
together = target_nll(probe_prompts, probe_targets)
alone = [target_nll([p], [t])[0] for p, t in zip(probe_prompts, probe_targets)]
assert np.allclose(together, alone, rtol=1e-4, atol=1e-3), (together, alone)
with torch.inference_mode():
    sequence = torch.tensor([probe_prompts[0] + probe_targets[0]], device=DEVICE)
    hf_labels = torch.tensor([[-100] * len(probe_prompts[0]) + probe_targets[0]], device=DEVICE)
    hf_loss = model(input_ids=sequence, labels=hf_labels).loss.item()
assert math.isclose(hf_loss * len(probe_targets[0]), alone[0], rel_tol=1e-4, abs_tol=1e-3), \
    (hf_loss * len(probe_targets[0]), alone[0])

word = tokenizer(" Hello", add_special_tokens=False)["input_ids"]
newline = tokenizer("\n", add_special_tokens=False)["input_ids"]
assert set(newline) <= NEWLINE_SET
assert finish(word + newline + word) == {"prediction": "Hello", "stop_reason": "newline",
                                         "generated_tokens": len(word) + 1}
assert finish(word + [EOS_ID, EOS_ID])["stop_reason"] == "eos"
assert finish(word * 3) == {"prediction": "Hello Hello Hello", "stop_reason": "cap",
                            "generated_tokens": 3 * len(word)}

probe_ids = [build_prompt(r["source"], DEMOS[CFG["demo_seeds"][0]][:example_shots],
                          CFG["max_new_tokens"])[0] for r in probe_rows]
probe_cap = min(16, CFG["max_new_tokens"])
batched = [x["prediction"] for x in generate_batch(probe_ids, probe_cap)]
single = [generate_batch([ids], probe_cap)[0]["prediction"] for ids in probe_ids]
agree = sum(a == b for a, b in zip(batched, single))
print("Tokenization, label alignment, cross-entropy, and stop-rule checks passed.")
print(f"Batched and one-at-a-time generation agree on {agree} of {len(single)} prompts.")
if agree < len(single) - 1:
    raise RuntimeError("Batched generation disagrees with one-at-a-time generation. "
                       "Check left padding and position IDs for this transformers version.")
if agree < len(single):
    print("One difference can come from a near-tied token. Inspect it if this repeats.")

## Run M0

### Translate and score one configuration

`run_config` builds one prompt per sentence, generates in batches sorted by prompt length, scores the references by teacher forcing when asked, and writes one JSON line per sentence to `runs/`. Each line keeps the ID, the source and reference, the demonstration IDs, the prompt and generated token counts, the stop reason, the translation, and the target NLL. When the IDs and the model commit match, a rerun reloads the file instead of generating again, so an interrupted run resumes at the next configuration. Batched times are listed for reference only. The benchmark section measures latency under controlled conditions.

In [ ]:
RUN_DIR = OUT / "runs"
RUN_DIR.mkdir(exist_ok=True)


def configurations():
    return [(0, None)] + [(k, seed) for k in CFG["shots"] if k > 0 for seed in CFG["demo_seeds"]]


def config_name(split, shots, seed):
    return f"{split}_{shots}shot" + ("" if seed is None else f"_seed{seed}")


def save_jsonl(path, records):
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(path)


def load_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def run_config(split, rows, shots, seed, with_ce):
    name = config_name(split, shots, seed)
    path, meta_path = RUN_DIR / f"{name}.jsonl", RUN_DIR / f"{name}.json"
    ids = [row["id"] for row in rows]
    if path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        records = load_jsonl(path)
        if ([r["id"] for r in records] == ids and meta["commit"] == COMMIT
                and (meta["with_ce"] or not with_ce)):
            return records, meta
    demos = DEMOS[seed][:shots] if shots else []
    prompts = [build_prompt(row["source"], demos, max(CFG["max_new_tokens"], len(row["target_ids"])))
               for row in rows]
    order = sorted(range(len(rows)), key=lambda i: -len(prompts[i][0]))
    outputs = [None] * len(rows)
    synchronize()
    start = time.perf_counter()
    for b in range(0, len(order), CFG["gen_batch_size"]):
        batch = order[b:b + CFG["gen_batch_size"]]
        for i, result in zip(batch, generate_batch([prompts[i][0] for i in batch], CFG["max_new_tokens"])):
            outputs[i] = result
    synchronize()
    gen_seconds = time.perf_counter() - start
    nll, ce_seconds = [None] * len(rows), None
    if with_ce:
        start = time.perf_counter()
        nll = target_nll([p for p, _ in prompts], [row["target_ids"] for row in rows])
        synchronize()
        ce_seconds = time.perf_counter() - start
    records = []
    for row, (prompt_ids, used), output, value in zip(rows, prompts, outputs, nll):
        records.append({
            "id": row["id"], "split": split, "shots": shots, "demo_seed": seed, "shots_used": used,
            "demo_ids": [d["id"] for d in demos[:used]], "source": row["source"],
            "reference": row["target"], "source_tokens": row["source_tokens"],
            "prompt_tokens": len(prompt_ids), "target_tokens": len(row["target_ids"]),
            "target_bytes": len((" " + row["target"]).encode("utf-8")), **output, "target_nll": value,
        })
    meta = {"name": name, "split": split, "shots": shots, "demo_seed": seed, "with_ce": with_ce,
            "commit": COMMIT, "n_sources": len(rows), "batch_size": CFG["gen_batch_size"],
            "gen_seconds": gen_seconds, "ce_seconds": ce_seconds, "device": hardware["gpu"] or str(DEVICE)}
    save_jsonl(path, records)
    write_json(meta_path, meta)
    return records, meta


def summarize(records, meta, scores):
    frame = pd.DataFrame(records)
    row = {
        "config": meta["name"], "split": meta["split"], "shots": meta["shots"],
        "demo_seed": meta["demo_seed"], "n_sources": len(frame),
        "bleu": scores["bleu"]["score"], "chrf": scores["chrf"]["score"],
        "mean_prompt_tokens": frame.prompt_tokens.mean(),
        "mean_generated_tokens": frame.generated_tokens.mean(),
        "pct_stop_newline": 100 * (frame.stop_reason == "newline").mean(),
        "pct_stop_eos": 100 * (frame.stop_reason == "eos").mean(),
        "pct_stop_cap": 100 * (frame.stop_reason == "cap").mean(),
        "rows_with_fewer_shots": int((frame.shots_used < frame.shots).sum()),
        "batched_ms_per_source": 1000 * meta["gen_seconds"] / len(frame),
    }
    if frame.target_nll.notna().all():
        total_nll = frame.target_nll.sum()
        row["ce_nats_per_token"] = total_nll / frame.target_tokens.sum()
        row["bits_per_byte"] = total_nll / math.log(2) / frame.target_bytes.sum()
    row["bleu_signature"] = scores["bleu"]["signature"]
    row["chrf_signature"] = scores["chrf"]["signature"]
    return row

### Choose the baseline configuration on validation

The M0 number that Q1 compares against should not be picked on the test set. This cell runs every configuration on a seeded sample of `select_sources` validation sentences and keeps the number of shots with the highest mean corpus chrF across demonstration seeds. Ties go to fewer shots. Below, that configuration is called the **chosen M0**. Every configuration is still reported on test.

In [ ]:
select_rows = random.Random(CFG["select_subset_seed"]).sample(
    splits["validation"], min(CFG["select_sources"], len(splits["validation"])))
selection_rows = []
for shots, seed in configurations():
    records, meta = run_config("validation", select_rows, shots, seed, with_ce=False)
    scores = corpus_scores([r["prediction"] for r in records], [r["reference"] for r in records])
    selection_rows.append(summarize(records, meta, scores))
    print(f"validation | {shots} shots | seed {seed} | chrF {scores['chrf']['score']:5.2f} | "
          f"BLEU {scores['bleu']['score']:5.2f}")
selection = pd.DataFrame(selection_rows)
selection.to_csv(OUT / "validation_selection.csv", index=False)
validation_chrf = selection.groupby("shots").chrf.mean()
CHOSEN_SHOTS = int(validation_chrf[validation_chrf == validation_chrf.max()].index.min())
display(validation_chrf.rename("mean validation chrF").to_frame().round(2))
print(f"Chosen M0: {CHOSEN_SHOTS} shots (mean chrF {validation_chrf.max():.2f} "
      f"on {len(select_rows)} validation sentences)")

### Score every configuration on the test split

Each configuration translates every test sentence and scores every reference by teacher forcing. Corpus BLEU and chrF compare the translations with the human references on a 0–100 scale, where higher is better. Target cross-entropy is the mean negative log-probability of the reference tokens, in nats per token, where lower is better. It measures how much probability GPT-2 assigns to the human translation, independently of the decoder. The interval columns hold 95% bootstrap intervals over test sentences.

In [ ]:
test_rows = splits["test"]
test_records, test_scores, run_rows = {}, {}, []
for shots, seed in configurations():
    records, meta = run_config("test", test_rows, shots, seed, with_ce=True)
    scores = corpus_scores([r["prediction"] for r in records], [r["reference"] for r in records])
    test_records[(shots, seed)], test_scores[(shots, seed)] = records, scores
    run_rows.append(summarize(records, meta, scores))
    print(f"test | {shots} shots | seed {seed} | chrF {scores['chrf']['score']:5.2f} | "
          f"BLEU {scores['bleu']['score']:5.2f} | CE {run_rows[-1]['ce_nats_per_token']:.3f}")

zero_key = (0, None)
COUNTS = bootstrap_counts(len(test_rows), CFG["bootstrap_samples"], CFG["bootstrap_seed"])
BOOT = {key: {name: bootstrap_scores(METRICS[name], scores[name]["stats"], COUNTS) for name in METRICS}
        for key, scores in test_scores.items()}
runs = pd.DataFrame(run_rows)
runs["demo_seed"] = runs["demo_seed"].astype("Int64")
for name in METRICS:
    bounds = [interval(BOOT[key][name]) for key in test_scores]
    runs[f"{name}_ci_low"] = [low for low, _ in bounds]
    runs[f"{name}_ci_high"] = [high for _, high in bounds]
runs.to_csv(OUT / "m0_runs.csv", index=False)
print("BLEU:", runs.bleu_signature.iloc[0])
print("chrF:", runs.chrf_signature.iloc[0])
display(runs[["shots", "demo_seed", "bleu", "bleu_ci_low", "bleu_ci_high", "chrf", "chrf_ci_low",
              "chrf_ci_high", "ce_nats_per_token", "bits_per_byte", "mean_prompt_tokens",
              "mean_generated_tokens", "pct_stop_cap", "rows_with_fewer_shots"]].round(3))

### Compare the numbers of shots (Q1 baseline)

The first table averages each metric over demonstration seeds and gives the sample standard deviation across seeds. That spread is the variability that comes from which training pairs are shown. The zero-shot row has a single run and no spread.

The second table pairs every few-shot configuration with zero-shot on the same test sentences. Δ is the seed-averaged difference in corpus points, positive when demonstrations help. The 95% interval and the bootstrap share of resamples with $\Delta \le 0$ come from the shared resamples, so they describe sentence-sampling uncertainty for these demonstration draws. The row marked `chosen` is the configuration selected on validation, which is the M0 baseline for Q1.

In [ ]:
metric_columns = ["bleu", "chrf", "ce_nats_per_token", "bits_per_byte", "mean_prompt_tokens",
                  "mean_generated_tokens", "pct_stop_cap", "batched_ms_per_source"]
by_k = runs.groupby("shots")[metric_columns].agg(["mean", "std"])
by_k.columns = [f"{metric}_{stat}" for metric, stat in by_k.columns]
by_k.insert(0, "runs", runs.groupby("shots").size())
by_k = by_k.reset_index()
by_k["chosen"] = by_k.shots == CHOSEN_SHOTS
by_k.to_csv(OUT / "m0_by_shots.csv", index=False)
display(by_k[["shots", "runs", "chosen", "bleu_mean", "bleu_std", "chrf_mean", "chrf_std",
              "ce_nats_per_token_mean", "ce_nats_per_token_std", "mean_prompt_tokens_mean",
              "mean_generated_tokens_mean", "pct_stop_cap_mean"]].round(3))

paired_rows = []
for k in [k for k in CFG["shots"] if k > 0]:
    keys = [(k, seed) for seed in CFG["demo_seeds"]]
    row = {"shots": k, "chosen": k == CHOSEN_SHOTS, "seeds": len(keys)}
    for name in METRICS:
        deltas = [test_scores[key][name]["score"] - test_scores[zero_key][name]["score"] for key in keys]
        boot_delta = np.mean([BOOT[key][name] - BOOT[zero_key][name] for key in keys], axis=0)
        low, high = interval(boot_delta)
        row.update({
            f"delta_{name}": float(np.mean(deltas)),
            f"delta_{name}_seed_sd": float(np.std(deltas, ddof=1)) if len(deltas) > 1 else float("nan"),
            f"delta_{name}_ci_low": low, f"delta_{name}_ci_high": high,
            f"share_boot_delta_{name}_le_0": float(np.mean(boot_delta <= 0)),
        })
    row["delta_ce_nats_per_token"] = float(runs.loc[runs.shots == k, "ce_nats_per_token"].mean()
                                           - runs.loc[runs.shots == 0, "ce_nats_per_token"].iloc[0])
    paired_rows.append(row)
paired = pd.DataFrame(paired_rows)
paired.to_csv(OUT / "m0_paired_vs_zero_shot.csv", index=False)
display(paired.round(3))

### Plot quality, cross-entropy, and prompt length

The left panel shows corpus chrF and BLEU against the number of shots, with bars of one standard deviation across demonstration seeds. The middle panel shows target cross-entropy. The right panel plots chrF against the mean prompt length in GPT-2 tokens, which is what the extra shots cost at inference time.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
for metric, label, color in [("chrf", "chrF", "tab:green"), ("bleu", "BLEU", "tab:blue")]:
    axes[0].errorbar(by_k.shots, by_k[f"{metric}_mean"], yerr=by_k[f"{metric}_std"].fillna(0),
                     marker="o", capsize=3, color=color, label=label)
axes[0].set(xlabel="Shots", ylabel="Corpus score", title="Translation quality")
axes[0].legend(fontsize=8)
axes[1].errorbar(by_k.shots, by_k["ce_nats_per_token_mean"], yerr=by_k["ce_nats_per_token_std"].fillna(0),
                 marker="o", capsize=3, color="tab:red")
axes[1].set(xlabel="Shots", ylabel="Nats per target token", title="Target cross-entropy")
axes[2].errorbar(by_k["mean_prompt_tokens_mean"], by_k["chrf_mean"], yerr=by_k["chrf_std"].fillna(0),
                 marker="o", capsize=3, color="tab:green")
for x_value, y_value, k in zip(by_k["mean_prompt_tokens_mean"], by_k["chrf_mean"], by_k.shots):
    axes[2].annotate(f"{k} shots", (x_value, y_value), xytext=(4, 6), textcoords="offset points",
                     fontsize=8)
axes[2].set(xlabel="Mean prompt length (GPT-2 tokens)", ylabel="Corpus chrF",
            title="Quality against prompt length")
for ax in axes[:2]:
    ax.set_xticks(list(by_k.shots))
for ax in axes:
    ax.grid(alpha=0.2)
fig.savefig(OUT / "m0_quality_ce_prompt.png", dpi=160, bbox_inches="tight")
plt.show()

### Relate cross-entropy to translation quality

Lower cross-entropy means GPT-2 finds the human translation more probable after this prompt. If that probability tracks quality, sentences with lower cross-entropy should also get higher sentence chrF. The table gives the Spearman rank correlation between the two for each number of shots (the Pearson correlation of ranks), averaged over demonstration seeds. The scatter shows the chosen M0 configuration for its first demonstration seed. Sentence chrF is noisy for short sentences, so the correlation describes these sentences rather than a calibrated relation.

In [ ]:
def sentence_frame(key):
    frame = pd.DataFrame(test_records[key])
    frame["sentence_chrf"] = [score_from_stats(CHRF_METRIC, row) for row in test_scores[key]["chrf"]["stats"]]
    frame["ce_per_token"] = frame.target_nll / frame.target_tokens
    return frame


def spearman(a, b):
    a, b = pd.Series(a, dtype=float), pd.Series(b, dtype=float)
    if a.nunique() < 2 or b.nunique() < 2:
        return float("nan")  # undefined when either variable is constant
    return float(a.rank().corr(b.rank()))


relation = pd.DataFrame([
    {"shots": key[0], "demo_seed": key[1],
     "spearman_ce_vs_chrf": spearman(frame.ce_per_token, frame.sentence_chrf)}
    for key, frame in ((key, sentence_frame(key)) for key in test_scores)])
relation["demo_seed"] = relation["demo_seed"].astype("Int64")
relation.to_csv(OUT / "m0_ce_vs_chrf.csv", index=False)
display(relation.groupby("shots").spearman_ce_vs_chrf.agg(["mean", "std"]).round(3))

chosen_key = zero_key if CHOSEN_SHOTS == 0 else (CHOSEN_SHOTS, CFG["demo_seeds"][0])
chosen_frame = sentence_frame(chosen_key)
fig, ax = plt.subplots(figsize=(5.5, 4.2), constrained_layout=True)
ax.scatter(chosen_frame.ce_per_token, chosen_frame.sentence_chrf, s=10, alpha=0.5)
ax.set(xlabel="Target cross-entropy (nats per token)", ylabel="Sentence chrF",
       title=f"Chosen M0, {CHOSEN_SHOTS} shots")
ax.grid(alpha=0.2)
fig.savefig(OUT / "m0_ce_vs_chrf.png", dpi=160, bbox_inches="tight")
plt.show()

## Q2 baseline: quality by source length

Q2(b) caps the training sentences at $L$ tokens and asks what happens on longer inputs. M0 was never trained on our data, so its curve shows how quality changes with length when no cap is involved. A drop beyond $L$ that M1 shows and M0 does not is a candidate effect of the cap. The bins use GPT-2 tokens of the German source. To put $L$ on a bin edge, edit `length_edges` in the settings, and use the same edges for M1. Each bin reports its sentence count, corpus chrF with a bootstrap interval, corpus BLEU, and target cross-entropy, averaged over demonstration seeds. Corpus BLEU on a small bin is unstable.

In [ ]:
bin_edges = [math.inf if edge is None else edge for edge in CFG["length_edges"]]
bin_labels = [f"{lo + 1}-{hi}" if math.isfinite(hi) else f"{lo + 1}+"
              for lo, hi in zip(bin_edges[:-1], bin_edges[1:])]
source_lengths = np.array([row["source_tokens"] for row in test_rows])
bin_of = np.digitize(source_lengths, bin_edges[1:-1], right=True)
bin_rows = []
for b, label in enumerate(bin_labels):
    members = np.flatnonzero(bin_of == b)
    if members.size == 0:
        continue
    bin_counts = bootstrap_counts(members.size, CFG["bootstrap_samples"], CFG["bootstrap_seed"] + 1 + b)
    for k in CFG["shots"]:
        keys = [zero_key] if k == 0 else [(k, seed) for seed in CFG["demo_seeds"]]
        values = {name: [score_from_stats(METRICS[name], test_scores[key][name]["stats"][members].sum(axis=0))
                         for key in keys] for name in METRICS}
        boot = np.mean([bootstrap_scores(CHRF_METRIC, test_scores[key]["chrf"]["stats"][members], bin_counts)
                        for key in keys], axis=0)
        low, high = interval(boot)
        ce = [sum(test_records[key][i]["target_nll"] for i in members)
              / sum(test_records[key][i]["target_tokens"] for i in members) for key in keys]
        bin_rows.append({"length_bin": label, "n_sentences": int(members.size), "shots": k,
                         "seeds": len(keys), "chrf": float(np.mean(values["chrf"])),
                         "chrf_ci_low": low, "chrf_ci_high": high,
                         "bleu": float(np.mean(values["bleu"])), "ce_nats_per_token": float(np.mean(ce))})
length_table = pd.DataFrame(bin_rows)
length_table.to_csv(OUT / "m0_length_bins.csv", index=False)
present = [label for label in bin_labels if label in set(length_table.length_bin)]
print("Sentences per bin:", length_table.groupby("length_bin").n_sentences.first().reindex(present).to_dict())
display(length_table.pivot(index="length_bin", columns="shots", values="chrf").reindex(present).round(2))

fig, ax = plt.subplots(figsize=(7, 4.2), constrained_layout=True)
positions = np.arange(len(present))
for k in CFG["shots"]:
    part = length_table[length_table.shots == k].set_index("length_bin").reindex(present)
    ax.plot(positions, part.chrf, marker="o", label=f"{k} shots")
    if k in {0, CHOSEN_SHOTS}:
        ax.fill_between(positions, part.chrf_ci_low, part.chrf_ci_high, alpha=0.15)
ax.set_xticks(positions)
ax.set_xticklabels(present)
ax.set(xlabel="German source length (GPT-2 tokens)", ylabel="Corpus chrF",
       title="M0 quality by source length")
ax.grid(alpha=0.2)
ax.legend(fontsize=8)
fig.savefig(OUT / "m0_length_bins.png", dpi=160, bbox_inches="tight")
plt.show()

## Q3 baseline: what demonstrations cost at inference time

Each demonstration lengthens the prompt, which costs prefill computation and KV-cache memory for every generated token. Demonstrations can also save work if they teach the model to stop after one line. This benchmark measures both effects on `eval_sources` test sentences drawn with seed `eval_subset_seed`, the M2 notebook's Q3 sample. In `tsv` mode these are exactly the sentences of M2's Q3(a) study.

**Protocol.** Each call decodes one source (batch size 1) with the KV cache on, which is the Hugging Face default, and with the same greedy decoder, cap, and stop rule, using the first demonstration seed. Every repeat shuffles the order of the shot counts, a short untimed warm-up runs first, and CUDA synchronization brackets each timed call. Latency runs from token IDs on the device to the decoded translation.

**Memory (GPU only).** For each call, the cell records the bytes of live tensors when `generate` returns, while its output still holds the KV cache, and the peak during the call. Both are measured relative to just before the call, and both read PyTorch's `requested_bytes` counters, which count tensor sizes without the caching allocator's block rounding. The prediction is `KV_BYTES_PER_POSITION` × (prompt tokens + generated tokens − 1), because the last generated token is never fed back. Held memory should match it closely, with one exception. GPT-2 computes queries, keys, and values in one matrix product, and right after the prompt pass the cache points into that product, which keeps the queries alive as well. The first decoding step copies the cache. A call that stops at its very first token therefore holds about 1.5 times the cache, and those calls are left out of the ratio. Peak memory also includes temporary activations and logits, whose size depends on the `transformers` version. Dilum's Q3(b) varies the batch size and switches the cache off, and it uses the same per-position constant.

In [ ]:
eval_rows = random.Random(CFG["eval_subset_seed"]).sample(test_rows, min(CFG["eval_sources"], len(test_rows)))
pd.DataFrame([{key: row[key] for key in ("id", "source", "target")} for row in eval_rows]).to_csv(
    OUT / "q3_test_subset.tsv", sep="\t", index=False)
cost_seed = CFG["demo_seeds"][0]
cost_prompts = {k: [build_prompt(row["source"], DEMOS[cost_seed][:k] if k else [],
                                 max(CFG["max_new_tokens"], len(row["target_ids"])))[0]
                    for row in eval_rows]
                for k in CFG["shots"]}
for k in CFG["shots"]:
    measured_generate(cost_prompts[k][0])  # untimed warm-up

cost_records = []
for repeat in range(CFG["timing_repeats"]):
    order = list(CFG["shots"])
    random.Random(7117 + repeat).shuffle(order)
    for k in order:
        for row, prompt_ids in zip(eval_rows, cost_prompts[k]):
            result, seconds, held, peak = measured_generate(prompt_ids)
            cached_positions = len(prompt_ids) + result["generated_tokens"] - 1
            cost_records.append({
                "repeat": repeat, "shots": k, "id": row["id"], "prompt_tokens": len(prompt_ids),
                "generated_tokens": result["generated_tokens"], "stop_reason": result["stop_reason"],
                "ms": 1000 * seconds, "predicted_kv_bytes": KV_BYTES_PER_POSITION * cached_positions,
                "held_bytes": held, "peak_bytes": peak, "prediction": result["prediction"]})
cost = pd.DataFrame(cost_records)
batched_lookup = {k: {r["id"]: r["prediction"] for r in test_records[zero_key if k == 0 else (k, cost_seed)]}
                  for k in CFG["shots"]}
cost["same_as_batched"] = [batched_lookup[k][i] == p
                           for k, i, p in zip(cost["shots"], cost["id"], cost["prediction"])]
cost["held_over_predicted"] = cost["held_bytes"] / cost["predicted_kv_bytes"]
# After a single generated token, the cache still points into the prefill's fused QKV output.
first_token_stops = int((cost["generated_tokens"] == 1).sum())
cost.loc[cost["generated_tokens"] == 1, "held_over_predicted"] = np.nan
cost.to_csv(OUT / "m0_cost.csv", index=False)
if DEVICE.type == "cuda":
    print(f"Calls left out of the held-memory ratio (stopped at the first token): {first_token_stops}")

latency = cost.groupby(["shots", "repeat"])["ms"].mean().groupby(level="shots").agg(["mean", "std"])
first_repeat = cost[cost["repeat"] == 0]
cost_summary = first_repeat.groupby("shots").agg(
    prompt_tokens=("prompt_tokens", "mean"), generated_tokens=("generated_tokens", "mean"),
    pct_stop_cap=("stop_reason", lambda s: 100 * (s == "cap").mean()),
    predicted_kv_mib=("predicted_kv_bytes", lambda s: s.mean() / 2**20),
    held_mib=("held_bytes", lambda s: s.mean() / 2**20),
    peak_mib=("peak_bytes", lambda s: s.mean() / 2**20),
    held_over_predicted=("held_over_predicted", "mean"),
    same_as_batched=("same_as_batched", "mean"))
cost_summary.insert(0, "ms_per_source_sd", latency["std"])
cost_summary.insert(0, "ms_per_source", latency["mean"])
cost_summary.to_csv(OUT / "m0_cost_summary.csv")
display(cost_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
axes[0].errorbar(cost_summary.prompt_tokens, cost_summary.ms_per_source,
                 yerr=cost_summary.ms_per_source_sd.fillna(0), marker="o", capsize=3)
for k, row in cost_summary.iterrows():
    axes[0].annotate(f"{k} shots", (row.prompt_tokens, row.ms_per_source), xytext=(4, 6),
                     textcoords="offset points", fontsize=8)
axes[0].set(xlabel="Mean prompt length (GPT-2 tokens)", ylabel="ms per source, batch size 1",
            title="Latency")
axes[1].plot(cost_summary.index, cost_summary.predicted_kv_mib, marker="o", label="predicted KV cache")
if DEVICE.type == "cuda":
    axes[1].plot(cost_summary.index, cost_summary.held_mib, marker="s", label="held after generate")
    axes[1].plot(cost_summary.index, cost_summary.peak_mib, marker="^", label="peak during generate")
axes[1].set_xticks(list(cost_summary.index))
axes[1].set(xlabel="Shots", ylabel="MiB per source", title="Memory beyond the weights")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.2)
fig.savefig(OUT / "m0_cost.png", dpi=160, bbox_inches="tight")
plt.show()

## Check the predictions recorded before scoring

Each line compares an observed value with the threshold that `prediction_before_scoring.json` stored before the first translation was scored. A prediction that fails is still a result for the report. The memory part of H5 needs a GPU and is marked as not measured otherwise.

In [ ]:
P = RECORDED["predictions"]
zero_run = runs.loc[runs.shots == 0].iloc[0]
by_shots = by_k.set_index("shots")
smallest = min(k for k in CFG["shots"] if k > 0)
largest = max(CFG["shots"])
checks = []


def check(name, observed, rule, met, note=""):
    checks.append({"prediction": name, "observed": None if observed is None else round(float(observed), 4),
                   "rule": rule, "met": None if met is None else bool(met), "note": note})


check("H1 zero-shot chrF", zero_run.chrf, f"<= {P['H1_zero_shot_chrf_max']}",
      zero_run.chrf <= P["H1_zero_shot_chrf_max"])
check("H1 zero-shot BLEU", zero_run.bleu, f"<= {P['H1_zero_shot_bleu_max']}",
      zero_run.bleu <= P["H1_zero_shot_bleu_max"])
gain = float(paired.loc[paired.shots == CHOSEN_SHOTS, "delta_chrf"].iloc[0]) if CHOSEN_SHOTS else 0.0
check("H2 chrF gain of the chosen configuration", gain, f">= {P['H2_min_chrf_gain_over_zero_shot']}",
      gain >= P["H2_min_chrf_gain_over_zero_shot"], f"chosen = {CHOSEN_SHOTS} shots")
drop = by_shots.loc[0, "ce_nats_per_token_mean"] - by_shots.loc[largest, "ce_nats_per_token_mean"]
check(f"H3 CE drop from 0 to {largest} shots", drop, f">= {P['H3_min_ce_drop_nats_per_token']}",
      drop >= P["H3_min_ce_drop_nats_per_token"])
rho = relation.loc[relation.shots == CHOSEN_SHOTS, "spearman_ce_vs_chrf"].mean()
check("H4 Spearman(CE, sentence chrF), chosen", rho, f"<= {P['H4_max_spearman_ce_vs_chrf']}",
      rho <= P["H4_max_spearman_ce_vs_chrf"] if np.isfinite(rho) else None,
      "" if np.isfinite(rho) else "undefined: a variable is constant")
reduction = 1 - (by_shots.loc[smallest, "mean_generated_tokens_mean"]
                 / by_shots.loc[0, "mean_generated_tokens_mean"])
check(f"H5 generated-token reduction, 0 to {smallest} shots", reduction,
      f">= {P['H5_min_generated_token_reduction']}", reduction >= P["H5_min_generated_token_reduction"])
if DEVICE.type == "cuda":
    worst = float((cost["held_over_predicted"] - 1).abs().max())
    check("H5 largest relative gap, held memory vs KV prediction", worst,
          f"<= {P['H5_held_memory_tolerance']}", worst <= P["H5_held_memory_tolerance"])
else:
    check("H5 held memory vs KV prediction", None, f"<= {P['H5_held_memory_tolerance']}", None,
          "not measured on CPU")
check_table = pd.DataFrame(checks)
write_json(OUT / "prediction_check.json", {
    "recorded_utc": RECORDED["recorded_utc"],
    "checked_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "checks": checks})
display(check_table)

## Error inspection

Automatic flags point to outputs worth reading. They are heuristics, not labels.

| Flag | Rule |
|---|---|
| empty output | nothing before the first stop token |
| copied source | identical to the source, or sentence chrF against the source of at least 60 |
| German output | at least two common German function words, and more German than English ones |
| omission | fewer than half as many words as a reference of at least four words |
| addition | more than twice as many words as the reference, plus two |
| names or numbers | the digit strings differ from the reference's, or a capitalized source word that the reference keeps is missing |
| template text | the output contains the `German:` or `English:` label |
| repetition | a word trigram occurs at least three times |
| runaway generation | the cap was reached before a newline or end-of-text |

The rate table gives the percentage of test outputs with each flag, averaged over demonstration seeds. The review sheet holds six seeded random sentences plus, for each flag, one flagged sentence under zero-shot and one under the chosen M0, with both systems' outputs for every sentence. After reading the source and the reference, fill in `manual_error_type` and `manual_notes` in `m0_error_review.csv`. The M2 notebook's review sheet uses the labels *copied source*, *omission*, *addition*, *wrong name or number*, *runaway generation*, and *acceptable paraphrase*.

In [ ]:
GERMAN_WORDS = set("""der die das und ist nicht ich sie es ein eine einen einem einer zu mit auf für
von den dem des wir auch sich wie aber oder wenn dass als noch nur mehr kann können haben hat sind
wird werden bei aus nach sehr hier diese dieser dieses uns mein meine wirklich jetzt dann schon immer
etwas keine kein gibt habe sein waren""".split())
ENGLISH_WORDS = set("""the and is not i you it to with on for of we how but or if that what only more
can have has are be at from after about very here this these they our my there which would because
really just do know think""".split())
assert not GERMAN_WORDS & ENGLISH_WORDS
FLAG_NAMES = ["empty output", "copied source", "German output", "omission", "addition",
              "names or numbers", "template text", "repetition", "runaway generation"]


def error_flags(source, reference, prediction, stop_reason):
    hyp_words = re.findall(r"\w+", prediction.casefold())
    ref_words = re.findall(r"\w+", reference.casefold())
    flags = []
    if not prediction:
        flags.append("empty output")
    else:
        if (prediction.casefold() == source.casefold()
                or CHRF_METRIC.sentence_score(prediction, [source]).score >= 60):
            flags.append("copied source")
        german = sum(w in GERMAN_WORDS for w in hyp_words)
        english = sum(w in ENGLISH_WORDS for w in hyp_words)
        if german >= 2 and german > english:
            flags.append("German output")
        if len(ref_words) >= 4 and len(hyp_words) < 0.5 * len(ref_words):
            flags.append("omission")
        if len(hyp_words) > 2 * len(ref_words) + 2:
            flags.append("addition")
        if f"{CFG['source_lang']}:" in prediction or f"{CFG['target_lang']}:" in prediction:
            flags.append("template text")
        trigrams = list(zip(hyp_words, hyp_words[1:], hyp_words[2:]))
        if trigrams and max(trigrams.count(t) for t in set(trigrams)) >= 3:
            flags.append("repetition")
    kept_names = ({w.strip(".,;:!?\"'()[]") for w in source.split()[1:] if w[:1].isupper()}
                  & set(re.findall(r"\w+", reference)))
    if (sorted(re.findall(r"\d+", reference)) != sorted(re.findall(r"\d+", prediction))
            or any(name not in prediction for name in kept_names)):
        flags.append("names or numbers")
    if stop_reason == "cap":
        flags.append("runaway generation")
    return flags


flag_rows = []
for (k, seed), records in test_records.items():
    for r in records:
        flags = error_flags(r["source"], r["reference"], r["prediction"], r["stop_reason"])
        flag_rows.append({"shots": k, "demo_seed": seed, "id": r["id"], "source": r["source"],
                          "reference": r["reference"], "prediction": r["prediction"],
                          "flags_to_check": "; ".join(flags), "any flag": bool(flags),
                          **{name: name in flags for name in FLAG_NAMES}})
flag_frame = pd.DataFrame(flag_rows)
flag_frame["demo_seed"] = flag_frame["demo_seed"].astype("Int64")
per_run = flag_frame.groupby(["shots", "demo_seed"], dropna=False)[FLAG_NAMES + ["any flag"]].mean()
error_rates = 100 * per_run.groupby(level="shots").mean()
error_rates.to_csv(OUT / "m0_error_rates.csv")
print("Percentage of test outputs with each flag:")
display(error_rates.round(1))


def outputs_for(key):
    k, seed = key
    part = flag_frame[flag_frame.shots == k]
    same_seed = part.demo_seed.isna() if seed is None else part.demo_seed.eq(seed).fillna(False)
    return part[same_seed.astype(bool)]


review_keys = [zero_key] + ([] if CHOSEN_SHOTS == 0 else [(CHOSEN_SHOTS, CFG["demo_seeds"][0])])
review_ids = random.Random(710).sample([row["id"] for row in test_rows], min(6, len(test_rows)))
reasons = {i: "random" for i in review_ids}
for key in review_keys:
    part = outputs_for(key)
    for j, name in enumerate(FLAG_NAMES):
        candidates = sorted(set(part.loc[part[name], "id"]) - set(review_ids))
        if candidates:
            pick = random.Random(711 + j).choice(candidates)
            review_ids.append(pick)
            reasons[pick] = f"flag: {name} (M0 {key[0]}-shot)"
indexed = {key: outputs_for(key).set_index("id") for key in review_keys}
review_rows = []
for i in review_ids:
    for key in review_keys:
        r = indexed[key].loc[i]
        review_rows.append({"selection": reasons[i], "id": i, "system": f"M0 {key[0]}-shot",
                            "demo_seed": key[1], "source": r["source"], "reference": r["reference"],
                            "prediction": r["prediction"], "flags_to_check": r["flags_to_check"],
                            "manual_error_type": "", "manual_notes": ""})
review = pd.DataFrame(review_rows)
review.to_csv(OUT / "m0_error_review.csv", index=False)
with pd.option_context("display.max_colwidth", 90):
    display(review[["selection", "id", "system", "source", "reference", "prediction",
                    "flags_to_check"]].head(12))

## Translate a new sentence with M0

Edit `NEW_SOURCE` and rerun the cell. It shows the zero-shot and chosen-configuration translations, then prints the full prompt that the chosen configuration reads. No reference is needed. A source that is too long for the context raises an error that states the limit.

In [ ]:
NEW_SOURCE = ("Anna kauft Brot." if PROFILE == "smoke" else
              "Wo ist der nächste Bahnhof?" if CFG["source_lang"] == "German" else
              splits["validation"][0]["source"])  # Replace with your own sentence.


def translate_m0(source, shots, seed=CFG["demo_seeds"][0]):
    source = clean_sentence(source)
    if not source:
        raise ValueError("Enter a nonempty source sentence.")
    prompt_ids, used = build_prompt(source, DEMOS[seed][:shots] if shots else [], CFG["max_new_tokens"])
    synchronize()
    start = time.perf_counter()
    result = generate_batch([prompt_ids], CFG["max_new_tokens"])[0]
    synchronize()
    return {"system": f"M0 {shots}-shot", "shots_used": used, "translation": result["prediction"],
            "stop_reason": result["stop_reason"], "prompt_tokens": len(prompt_ids),
            "ms": round(1000 * (time.perf_counter() - start), 1)}, prompt_ids


demo_rows, demo_prompts = [], {}
for k in sorted({0, CHOSEN_SHOTS}):
    demo_row, demo_prompts[k] = translate_m0(NEW_SOURCE, k)
    demo_rows.append(demo_row)
print(NEW_SOURCE)
display(pd.DataFrame(demo_rows))
print(f"Prompt read by the chosen M0 ({CHOSEN_SHOTS} shots):\n")
print(tokenizer.decode(demo_prompts[CHOSEN_SHOTS]))

## Collect the files for the team and the report

`m0_results_table.tex` is a booktabs table for the report, with one row per number of shots. Entries are means with standard deviations over demonstration seeds where there is more than one run, and the chosen configuration is in bold. Add `\usepackage{booktabs}` to the preamble and include the table with `\input{m0_results_table.tex}`. `m0_chosen_test_predictions.tsv` lists the zero-shot and chosen-M0 test translations by ID, which M1 needs for paired comparisons. The ZIP holds the whole run folder. The notebook itself can be saved from Colab's **File → Download → Download .ipynb**.

| File | Use |
|---|---|
| `settings.json`, `model_info.json`, `data_report.json`, `demonstrations.json` | Configuration, versions, model commit, data fingerprint, and the demonstrations |
| `prediction_before_scoring.json`, `prediction_check.json` | Predictions with their timestamp, and the check |
| `runs/*.jsonl` | Every translation, stop reason, token count, and target NLL, by ID |
| `m0_runs.csv`, `m0_by_shots.csv`, `m0_paired_vs_zero_shot.csv` | Scores, signatures, intervals, and paired differences |
| `validation_selection.csv` | How the chosen M0 was selected |
| `m0_ce_vs_chrf.csv`, `m0_length_bins.csv` | Cross-entropy against quality, and the Q2 baseline by source length |
| `m0_cost.csv`, `m0_cost_summary.csv`, `q3_test_subset.tsv` | Q3 baseline: latency and memory per call, and the benchmark sentences |
| `m0_error_rates.csv`, `m0_error_review.csv` | Error flags and the manual review sheet |
| `*.png` | Figures |

In [ ]:
def table_cell(mean, sd, digits):
    text = f"{mean:.{digits}f}"
    return text if not np.isfinite(sd) else f"{text} $\\pm$ {sd:.{digits}f}"


latency_by_shots = cost_summary["ms_per_source"].to_dict()
lines = [r"\begin{tabular}{@{}rcccccc@{}}", r"\toprule",
         r"Shots & BLEU & chrF & CE (nats/token) & Prompt tokens & Generated tokens & ms/source \\",
         r"\midrule"]
for _, row in by_k.iterrows():
    cells = [str(int(row.shots)), table_cell(row.bleu_mean, row.bleu_std, 2),
             table_cell(row.chrf_mean, row.chrf_std, 2),
             table_cell(row.ce_nats_per_token_mean, row.ce_nats_per_token_std, 3),
             f"{row.mean_prompt_tokens_mean:.0f}", f"{row.mean_generated_tokens_mean:.1f}",
             f"{latency_by_shots.get(int(row.shots), float('nan')):.0f}"]
    if row.chosen:
        cells = [f"\\textbf{{{cell}}}" for cell in cells]
    lines.append(" & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
(OUT / "m0_results_table.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")

shared_keys = [zero_key] + ([] if CHOSEN_SHOTS == 0 else [(CHOSEN_SHOTS, seed) for seed in CFG["demo_seeds"]])
chosen_table = pd.DataFrame({"id": [row["id"] for row in test_rows],
                             "source": [row["source"] for row in test_rows],
                             "reference": [row["target"] for row in test_rows]})
for key in shared_keys:
    column = f"m0_{key[0]}shot" + ("" if key[1] is None else f"_seed{key[1]}")
    chosen_table[column] = [r["prediction"] for r in test_records[key]]
chosen_table.to_csv(OUT / "m0_chosen_test_predictions.tsv", sep="\t", index=False)

archive = Path(shutil.make_archive(str(OUT.parent / f"{OUT.name}_results"), "zip", root_dir=OUT))
print("\n".join(lines))
print(f"\nResults directory: {OUT}")
print(f"Downloadable ZIP: {archive} ({archive.stat().st_size / 2**20:.1f} MiB)")
if "google.colab" in sys.modules:
    from google.colab import files
    DOWNLOAD_ZIP = False  # Set True and rerun this cell to download the ZIP in the browser.
    if DOWNLOAD_ZIP:
        files.download(str(archive))

### Notes for the report

| Topic | What to state |
|---|---|
| The system | Pretrained GPT-2 small (124,439,808 parameters) with frozen weights, the shared prompt with 0 to 8 demonstrations from the training split, greedy decoding, and a newline stop. Quote the commit in `model_info.json`. |
| Token budgets | M0 trains on no tokens of our data. Its budget is the prompt, so report mean prompt tokens per shot count next to the training-token budgets of M1 and M2. GPT-2 was pretrained on WebText, about 40 GB of text (Radford et al., 2019). |
| Q1 | Quote the chosen M0 row on test with its bootstrap interval and seed spread, and state the validation rule that picked it. Compare M1 with it by paired tests on the IDs in `m0_chosen_test_predictions.tsv`. |
| Q2 and Q3 | Quote the length-bin curve next to M1's, and the latency and held-memory columns next to Dilum's batch-size study. |
| Variability | Two sources are reported separately: the choice of demonstrations (standard deviation over seeds) and the sampling of test sentences (bootstrap). Greedy decoding has no sampling noise, so repeated timing runs only measure timing spread. |
| Limitations | GPT-2 saw little German, and the newline stop rule is part of the method. The bootstrap intervals hold the demonstration draws fixed, and only one test domain (TED talks) is used. In `iwslt` mode the rows can differ slightly from M2's split, which `tsv` mode avoids. |